# Load the CEOL Universe and take a first look

Every row of the CEOL Universe is a candidate string, with its provenance. A
row says which published lists and corpora hold the string and how it entered
the pool. It does not say the string is a word.

This notebook loads version 1.0 of the dataset (here, the fixtures in
`kit/fixtures/`), reads the evidence columns correctly, and compares the
fixtures with the counts the release publishes. It prints counts only and
never a candidate.

In [ ]:
import os
import sys
from pathlib import Path

# kit/ holds the ceolkit package, and this notebook sits in kit/notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from ceolkit import figure, load, strata

# The fixtures are a sample of version 1.0's rows (kit/fixtures/). To use a
# full config, download its file and point SOURCE at it; with SOURCE = None,
# load() reads the hub, if the dataset is public or you are signed in to it.
SOURCE = Path(os.environ.get("CEOL_FIXTURES", "../fixtures"))

## Load a config

`load(config, version, columns=None, source=None)` reads one config at a
pinned version and returns an Arrow table. The release has `universe` (every
row), `features` (the corpus features of the single-token rows), and a set of
filters of `universe`: `panel-listed`, `eol-valid`, `spaced`, `exceptions`, and
`sample`. A released version never changes, so a version and a config name one
file. Each file carries its version in its
metadata, and `load` refuses a file of another version.

The fixtures hold 2 files, a sample of `universe` and the matching rows of
`features`.

In [ ]:
universe = load("universe", "v1.0", source=SOURCE)
features = load("features", "v1.0", source=SOURCE)

print(f"universe sample: {universe.num_rows} rows, {universe.num_columns} columns")
print(f"features sample: {features.num_rows} rows, {features.num_columns} columns")
print("version in the file's metadata:", universe.schema.metadata[b"ceol.version"].decode())

# From the hub, once the dataset is public (or in an environment signed in to it):
# universe = load("universe", "v1.0")

## What the release publishes

The pipeline writes the release's counts to `figures.json`, each under an id.
`figure(id)` reads one, so a count in a document or a notebook is read and not
typed.

In [ ]:
configs = ["universe", "features", "panel-listed", "eol-valid", "spaced", "exceptions", "sample"]
pd.DataFrame({
    "config": configs,
    "rows in the release": [figure(f"config.{name}.rows") for name in configs],
})

In [ ]:
print("columns in universe:", figure("columns.universe"), "; columns in features:", figure("columns.features"))
print("columns of the universe sample match:", universe.num_columns == figure("columns.universe"))

## The fixtures are not the universe

The fixtures hold the same number of rows from each intake layer. Their
proportions are not the release's, so a share computed on them describes the
fixtures only. For an estimate about the release, draw a stratified sample and
weight it (`02_draw_a_labeling_sample.ipynb`).

In [ ]:
frame = universe.to_pandas()
layers = frame["intake_layer"].value_counts().sort_index()
pd.DataFrame({
    "rows in the fixtures": layers,
    "rows in the release": [figure(f"rows.by_intake_layer.{name}") for name in layers.index],
})

## The evidence columns: 1, 0, -1, and null

`wordnet`, `wiktionary_english`, `hunspell_en_US`, `hunspell_en_GB`,
`hunspell_en_CA`, `hunspell_en_AU`, and `enable1` hold one of 4 things:

- **1**: the source lists the row.
- **0**: the source does not list it.
- **-1**: the source cannot hold this kind of string. A list without hyphens
  cannot hold a hyphenated string. This is not "not listed".
- **null**: the source was not measured over the row. Null is never "not
  listed" either.

`scowl_tier` is the size tier (10 to 95) of the smallest SCOWL list that holds
the string, 0 for not listed, -1 for not applicable, and null for not measured.

In [ ]:
evidence = ["wordnet", "wiktionary_english", "hunspell_en_US", "hunspell_en_GB",
            "hunspell_en_CA", "hunspell_en_AU", "enable1", "scowl_tier"]


def evidence_counts(column):
    cells = frame[column]
    listed = (cells > 0) if column == "scowl_tier" else (cells == 1)
    return {"listed": int(listed.sum()),
            "not listed (0)": int((cells == 0).sum()),
            "cannot hold (-1)": int((cells == -1).sum()),
            "not measured (null)": int(cells.isna().sum())}


pd.DataFrame({column: evidence_counts(column) for column in evidence}).T

Null appears only on rows of more than 1 token, because the single-token
sources were not measured over them. A -1 appears on single-token rows, where
the string's class is one the source cannot hold.

In [ ]:
has_null = frame[evidence].isna().any(axis=1)
has_minus_1 = (frame[evidence] == -1).any(axis=1)
single = frame["token_count"] == 1

print("rows with a null evidence cell:", int(has_null.sum()))
print("  of them, rows of more than 1 token:", int((has_null & ~single).sum()))
print("single-token rows with a -1 in some evidence column:", int((has_minus_1 & single).sum()))

A -1 follows the spelling class. This table counts single-token rows by their
spelling class (see `strata.spelling_class`) and by ENABLE's cell.

In [ ]:
singles = frame[single].copy()
singles["spelling_class"] = strata.spelling_class.label_frame(singles)
pd.crosstab(singles["spelling_class"], singles["enable1"].map({-1: "-1", 0: "0", 1: "1"}))

## The evidence tier

`strata.evidence_tier` turns the evidence columns into one label for each
single-token row: listed by 2 or more of the 5 list families, by 1, by none
with corpus containment, or by none and no corpus. The definition says how it
treats -1 and null.

In [ ]:
print(strata.evidence_tier.definition)

In [ ]:
tiers = pd.Series(strata.evidence_tier.label_frame(frame)).value_counts()
tiers.reindex([t for t in strata.evidence_tier.labels if t in tiers.index])

## Provenance and the model or engine

`provenance_class` says whether a row has a published intake channel and
whether a published list or a corpus holds it. `model_or_engine` records how
the English OpenList decided a row: by its synthetic engine, or on its valid
list through a language model. It is a record of how the EOL decided and not a
verdict of this dataset. `CONTAMINATION.md` says how to use it.

In [ ]:
for column in ("provenance_class", "model_or_engine"):
    counts = frame[column].value_counts().sort_index()
    print(column)
    print(pd.DataFrame({
        "rows in the fixtures": counts,
        "rows in the release": [figure(f"rows.by_{column}.{name}") for name in counts.index],
    }), "\n")

## Next

- `02_draw_a_labeling_sample.ipynb` draws a stratified sample for a labeling
  study and saves its manifest.
- `03_frequency_and_coverage.ipynb` looks at frequency bands and at coverage by
  list family.